# ATLAS-Q: GPU-Accelerated Quantum Tensor Network Simulator
**Adaptive Tensor Learning And Simulation – Quantum**

**Version 0.5.0** | **October 2025**

This notebook provides a complete, interactive demonstration of ATLAS-Q features.

---

## What is ATLAS-Q?

ATLAS-Q is a GPU-accelerated quantum simulator featuring:

- **77K+ ops/sec** gate throughput (GPU-optimized)
- **626,000× memory compression** vs full statevector (30 qubits)
- **20× speedup** on Clifford circuits (Stabilizer backend)
- **Custom Triton kernels** for 1.5-3× gate operation speedup

**Key Capabilities:**
1. Adaptive Matrix Product States (MPS) for efficient simulation
2. Period-finding & integer factorization (Shor's algorithm)
3. NISQ noise models for realistic simulation
4. Stabilizer formalism for fast Clifford circuits
5. VQE/QAOA variational algorithms
6. Time evolution with TDVP
7. 2D circuit support with automatic SWAP insertion

---

## 📚 Related Documentation

- **[Complete Guide](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/COMPLETE_GUIDE.md)** - Full API reference and tutorials
- **[Feature Status](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/FEATURE_STATUS.md)** - What's implemented
- **[GitHub Repository](https://github.com/followthsapper/ATLAS-Q)** - Source code and issues

---
## 1. Installation

### For Google Colab
Run this cell to install ATLAS-Q in Colab:

In [ ]:
# Uncomment to install in Google Colab
# !pip install atlas-quantum

# For GPU support (if Colab GPU is available)
# !pip install atlas-quantum[gpu]

### For Local Jupyter
If running locally, install via:
```bash
pip install atlas-quantum[gpu]  # With GPU support
# or
pip install atlas-quantum       # CPU only
```

### Check GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = 'cpu'
    print("⚠️  No GPU detected, using CPU")
    print("   (Performance will be slower but all features work)")

print(f"\nUsing device: {device}")

### Verify Installation

In [ ]:
import atlas_q
from atlas_q import get_quantum_sim

print(f"ATLAS-Q version: {atlas_q.__version__}")

# Test basic functionality
QCH, _, _, _ = get_quantum_sim()
sim = QCH()
print("✅ Installation verified - ATLAS-Q is ready!")

---
## 2. Quick Start Examples

### Example 1: Factor a Number (Shor's Algorithm)

In [ ]:
from atlas_q import get_quantum_sim

# Get simulator
QCH, _, _, _ = get_quantum_sim()
sim = QCH()

# Factor 221 using quantum period-finding
factors = sim.factor_number(221)
print(f"221 = {factors[0]} × {factors[1]}")

# Try other semiprimes
for N in [15, 21, 143]:
    factors = sim.factor_number(N)
    print(f"{N} = {factors[0]} × {factors[1]}")

**How it works:** ATLAS-Q uses Shor's algorithm with compressed quantum states:
- Periodic states: O(1) memory (not O(2^n))
- Product states: O(n) memory
- Verified against IBM (N=15), photonic (N=21), NMR (N=143) benchmarks

### Example 2: Simulate 10 Qubits with Adaptive MPS

In [ ]:
import torch
from atlas_q import get_adaptive_mps

# Get MPS modules
mps_modules = get_adaptive_mps()
AdaptiveMPS = mps_modules['AdaptiveMPS']

# Create 10-qubit system
mps = AdaptiveMPS(10, bond_dim=8, device=device)

# Apply Hadamard gates
H = torch.tensor([[1,1],[1,-1]], dtype=torch.complex64) / torch.sqrt(torch.tensor(2.0))
H = H.to(device)

for q in range(10):
    mps.apply_single_qubit_gate(q, H)

# Check statistics
stats = mps.stats_summary()
print(f"Max bond dimension: {stats['max_chi']}")
print(f"Memory usage: {mps.memory_usage() / 1024:.2f} KB")
print(f"Total operations: {stats['total_operations']}")

**Memory efficiency:** Full statevector for 10 qubits = 2^10 × 16 bytes = 16 KB

MPS representation: ~2 KB (8× compression for this simple case)

For 30 qubits: 16 GB → 0.03 MB (626,000× compression!)

### Example 3: Build and Use Hamiltonians

In [ ]:
from atlas_q import get_mpo_ops, get_adaptive_mps
import torch

# Get modules
mpo_modules = get_mpo_ops()
mps_modules = get_adaptive_mps()

# Build Ising Hamiltonian: H = -J Σ Z_i Z_{i+1} - h Σ X_i
MPOBuilder = mpo_modules['MPOBuilder']
H = MPOBuilder.ising_hamiltonian(n_sites=6, J=1.0, h=0.5, device=device)

# Create MPS state
AdaptiveMPS = mps_modules['AdaptiveMPS']
mps = AdaptiveMPS(6, bond_dim=8, device=device)

# Apply gates to create superposition
H_gate = torch.tensor([[1,1],[1,-1]], dtype=torch.complex64) / torch.sqrt(torch.tensor(2.0))
H_gate = H_gate.to(device)
for q in range(6):
    mps.apply_single_qubit_gate(q, H_gate)

# Compute expectation value <ψ|H|ψ>
expectation_value = mpo_modules['expectation_value']
energy = expectation_value(H, mps)
print(f"Energy <ψ|H|ψ> = {energy.real:.6f}")

**Available Hamiltonians:**
- Ising: `ising_hamiltonian(n_sites, J, h)`
- Heisenberg: `heisenberg_hamiltonian(n_sites, Jx, Jy, Jz)`
- Custom: Build your own MPO

### Example 4: Add Noise to Circuit (NISQ Simulation)

In [ ]:
from atlas_q import get_noise_models, get_adaptive_mps
import torch

# Get modules
noise_modules = get_noise_models()
mps_modules = get_adaptive_mps()

# Create depolarizing noise model (p=0.1% per gate)
NoiseModel = noise_modules['NoiseModel']
noise = NoiseModel.depolarizing(p1q=0.001, device=device)

# Create MPS
AdaptiveMPS = mps_modules['AdaptiveMPS']
mps = AdaptiveMPS(5, bond_dim=4, device=device)

# Apply noisy gates
StochasticNoiseApplicator = noise_modules['StochasticNoiseApplicator']
applicator = StochasticNoiseApplicator(noise, seed=42)

H = torch.tensor([[1,1],[1,-1]], dtype=torch.complex64) / torch.sqrt(torch.tensor(2.0))
H = H.to(device)

for i in range(10):
    mps.apply_single_qubit_gate(0, H)
    applicator.apply_1q_noise(mps, 0)

# Check fidelity degradation
fidelity = applicator.get_fidelity_estimate()
print(f"Estimated fidelity after 10 noisy gates: {fidelity:.4f}")
print(f"Expected fidelity: {(1 - 0.001)**10:.4f}")

**Noise Models:**
- Depolarizing: `NoiseModel.depolarizing(p1q, p2q)`
- Amplitude damping: `NoiseModel.amplitude_damping(gamma)`
- Phase damping: `NoiseModel.phase_damping(lambda_)`
- Custom: Build using Kraus operators

### Example 5: Fast Clifford Simulation (20× Speedup)

In [ ]:
from atlas_q import get_stabilizer
import time

# Get stabilizer modules
stab_modules = get_stabilizer()
StabilizerSimulator = stab_modules['StabilizerSimulator']

# Create simulator for 50 qubits (only 50×50 tableau storage!)
sim = StabilizerSimulator(n_qubits=50)

# Apply Clifford gates (very fast!)
start = time.time()
for i in range(1000):
    sim.h(i % 50)        # Hadamard
    sim.s(i % 50)        # S gate
    if i < 999:
        sim.cnot(i % 50, (i+1) % 50)  # CNOT
elapsed = time.time() - start

# Measure
outcome = sim.measure(qubit=0)
print(f"Measurement outcome: {outcome}")
print(f"Applied 2000 gates in {elapsed:.3f}s ({2000/elapsed:.0f} ops/sec)")
print(f"Memory: O(n²) = O(50²) = 2500 entries (not 2^50!)")

**Why so fast?**
- Stabilizer formalism tracks stabilizers, not quantum state
- Memory: O(n²) instead of O(2^n)
- Gates: O(n²) instead of O(2^n)
- Works for Clifford gates: H, S, CNOT, CZ, SWAP

---
## 3. Advanced Features

### 3.1 Variational Quantum Eigensolver (VQE)

In [ ]:
from atlas_q.vqe_qaoa import VQE, VQEConfig
from atlas_q import get_mpo_ops

# Build Heisenberg Hamiltonian
mpo_modules = get_mpo_ops()
MPOBuilder = mpo_modules['MPOBuilder']
H = MPOBuilder.heisenberg_hamiltonian(n_sites=6, Jx=1.0, Jy=1.0, Jz=1.0, device=device)

# Configure VQE
config = VQEConfig(
    n_layers=3,
    max_iter=50,
    bond_dim=8,
    device=device
)

# Run VQE optimization
vqe = VQE(H, config)
energy, params = vqe.run()

print(f"Ground state energy: {energy:.6f}")
print(f"Optimized parameters: {len(params)} values")

### 3.2 Time Evolution with TDVP

In [ ]:
from atlas_q.tdvp import TDVP1Site, TDVPConfig
from atlas_q import get_mpo_ops, get_adaptive_mps
import matplotlib.pyplot as plt

# Create Hamiltonian
mpo_modules = get_mpo_ops()
MPOBuilder = mpo_modules['MPOBuilder']
H = MPOBuilder.ising_hamiltonian(n_sites=10, J=1.0, h=0.5, device=device)

# Create initial state
mps_modules = get_adaptive_mps()
AdaptiveMPS = mps_modules['AdaptiveMPS']
mps = AdaptiveMPS(10, bond_dim=8, device=device)

# Configure TDVP
config = TDVPConfig(
    dt=0.01,
    t_final=1.0,
    use_gpu_optimized=True if device == 'cuda' else False
)

# Run time evolution
tdvp = TDVP1Site(H, mps, config)
times, energies = tdvp.run()

# Plot energy vs time
plt.figure(figsize=(10, 4))
plt.plot(times, [e.real for e in energies])
plt.xlabel('Time')
plt.ylabel('Energy')
plt.title('TDVP Time Evolution')
plt.grid(True)
plt.show()

print(f"Final energy: {energies[-1].real:.6f}")

### 3.3 2D Circuit Simulation

In [ ]:
from atlas_q import get_planar_2d
import torch

# Get 2D circuit modules
planar_modules = get_planar_2d()
Planar2DCircuit = planar_modules['Planar2DCircuit']

# Create 3×3 grid (9 qubits)
circuit = Planar2DCircuit(grid_shape=(3, 3), device=device)

# Apply nearest-neighbor gates
CNOT = torch.tensor([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]],
                     dtype=torch.complex64).to(device)

# Apply gate between (0,0) and (0,1)
circuit.apply_2d_gate((0,0), (0,1), CNOT)

# Apply gate between (0,0) and (1,0) - different direction
circuit.apply_2d_gate((0,0), (1,0), CNOT)

# Check statistics
stats = circuit.get_stats()
print(f"Total gates applied: {stats['total_gates']}")
print(f"SWAP gates inserted: {stats['swap_count']}")

**2D Circuit Features:**
- Automatic SWAP insertion for non-nearest-neighbor gates
- Optimized routing for grid connectivity
- Supports any rectangular grid topology

---
## 4. Performance Benchmarks

### Memory Comparison: MPS vs Statevector

In [ ]:
from atlas_q import get_adaptive_mps
import torch

mps_modules = get_adaptive_mps()
AdaptiveMPS = mps_modules['AdaptiveMPS']

n_qubits_list = [10, 15, 20, 25, 30]

print("Qubits | Statevector Memory | MPS Memory  | Compression")
print("-------|-------------------|-------------|------------")

for n in n_qubits_list:
    # Statevector memory
    statevector_bytes = 2**n * 16  # complex128 = 16 bytes
    
    # MPS memory (approximate)
    mps = AdaptiveMPS(n, bond_dim=8, device='cpu')
    mps_bytes = mps.memory_usage()
    
    compression = statevector_bytes / mps_bytes
    
    sv_str = f"{statevector_bytes/1e6:.1f} MB" if statevector_bytes < 1e9 else f"{statevector_bytes/1e9:.1f} GB"
    mps_str = f"{mps_bytes/1e3:.1f} KB" if mps_bytes < 1e6 else f"{mps_bytes/1e6:.2f} MB"
    
    print(f"{n:6d} | {sv_str:17s} | {mps_str:11s} | {compression:.0f}×")

print("\n✅ MPS achieves massive compression for low-entanglement states!")

### Speed Comparison: Stabilizer vs MPS for Clifford Circuits

In [ ]:
from atlas_q import get_stabilizer, get_adaptive_mps
import torch
import time

n_qubits = 20
n_gates = 1000

# Stabilizer simulation
stab_modules = get_stabilizer()
StabilizerSimulator = stab_modules['StabilizerSimulator']
sim = StabilizerSimulator(n_qubits=n_qubits, device=device)

start = time.time()
for i in range(n_gates):
    sim.h(i % n_qubits)
    if i < n_gates - 1:
        sim.cnot(i % n_qubits, (i+1) % n_qubits)
stab_time = time.time() - start

# MPS simulation (same circuit)
mps_modules = get_adaptive_mps()
AdaptiveMPS = mps_modules['AdaptiveMPS']
mps = AdaptiveMPS(n_qubits, bond_dim=4, device=device)

H = torch.tensor([[1,1],[1,-1]], dtype=torch.complex64) / torch.sqrt(torch.tensor(2.0))
CNOT = torch.tensor([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=torch.complex64).reshape(2,2,2,2)
H = H.to(device)
CNOT = CNOT.to(device)

start = time.time()
for i in range(n_gates):
    mps.apply_single_qubit_gate(i % n_qubits, H)
    if i < n_gates - 1:
        mps.apply_two_site_gate(i % n_qubits, CNOT)
mps_time = time.time() - start

print(f"Stabilizer: {n_gates} gates in {stab_time:.3f}s ({n_gates/stab_time:.0f} ops/sec)")
print(f"MPS:        {n_gates} gates in {mps_time:.3f}s ({n_gates/mps_time:.0f} ops/sec)")
print(f"\nSpeedup: {mps_time/stab_time:.1f}× faster with Stabilizer!")

---
## 5. What's NOT Implemented

To be completely honest, here are features mentioned in other quantum frameworks but NOT in ATLAS-Q:

❌ **Molecular Hamiltonians from specifications** - No automatic H2, LiH builders
❌ **MaxCut Hamiltonian builder** - Need to build manually
❌ **Full PEPS implementation** - Only basic support
❌ **Multi-GPU distributed MPS** - Single GPU only
❌ **Qiskit/Cirq circuit import** - Coming soon

See [FEATURE_STATUS.md](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/FEATURE_STATUS.md) for complete status.

---
## 6. Troubleshooting

### Issue: "triton_kernels not found"
**Solution:** Install Triton: `pip install triton>=2.0.0`

### Issue: "ptxas fatal: Value 'sm_XXX' is not defined"
**Solution:** Set GPU architecture:
```bash
export TORCH_CUDA_ARCH_LIST="8.0;9.0;12.0"
export TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas"
```

### Issue: Out of memory on GPU
**Solution:** Reduce `bond_dim` or switch to CPU:
```python
mps = AdaptiveMPS(n_sites, bond_dim=4, device='cpu')  # Smaller bond_dim
```

### Issue: Slow performance on CPU
**Solution:** Expected - CPU is 10-100× slower than GPU. Use Stabilizer backend for Clifford circuits:
```python
from atlas_q import get_stabilizer
# 20× faster for H, S, CNOT, CZ gates
```

---
## 7. Next Steps

### Learn More
- **[Complete Guide](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/COMPLETE_GUIDE.md)** - Full API reference
- **[Research Paper](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/RESEARCH_PAPER.md)** - Mathematical foundations
- **[Whitepaper](https://github.com/followthsapper/ATLAS-Q/blob/main/docs/WHITEPAPER.md)** - Architecture details

### Contributing
See [CONTRIBUTING.md](https://github.com/followthsapper/ATLAS-Q/blob/main/CONTRIBUTING.md) for guidelines.

### Issues & Support
- **Issues:** [GitHub Issues](https://github.com/followthsapper/ATLAS-Q/issues)
- **Discussions:** [GitHub Discussions](https://github.com/followthsapper/ATLAS-Q/discussions)

---

**ATLAS-Q**: Making quantum simulation accessible through honest, working code.

**License:** MIT | **Version:** 0.5.0 | **Updated:** October 2025